# 1. Introduction

This notebook explains how to work with the linked masses model.

## Installation and Running

Activate the project `LinkedMasses`:

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))

When running the project for the first time or after updating the source code from Git, run `Pkg.instantiate()` to resolve package dependencies.

In [ ]:
Pkg.instantiate()

## The Linked-Masses Model

Model of two point masses connected by a rigid link. The two masses are subjected to linear damping and environmental disturbances in the form of ocean currents. In this project, the linked masses are used to simulate a fully actuated autonomous surface vehicle (ASV) towing a payload. The parameters of the model are encapsulated in `LinkedMassesParameters`.

Most functions and structures in this packages have a docstring. Use Julia's help menu (`?`) to learn more about them.

In [ ]:
using LinkedMasses
@doc LinkedMassesParameters

Let us create a simplified model of the Wave Adaptive Modular Vehicle (WAMV) with parameters taken from (https://github.com/osrf/vrx). The mass of the vehicle, including batteries and sensors, is around 200 kg. The damping coefficient is around 250 Ns/m. Suppose that the WAMV is towing a 40kg payload with a damping coefficient of 50 Ns/m on a 10m long cable:

In [ ]:
m0 = 200.0
m = 40.0
c0 = 250.0
c = 50.0
L = 10.0

params = LinkedMassesParameters(L, m0, m, c0, c)

## ODEs

The linked-masses model has 3 degrees of freedom - position of the first mass, and the angle of the link ($q = [x_0, y_0, \theta]$).
The full state consists of the generalized coordinates and velocities - $x = [q, \dot{q}] \in \mathbb{R}^6$.

Use `linked_masses_ode` to calculate the derivative of the state:

In [ ]:
@doc linked_masses_ode

### Example 1

Let us apply a force of 500 newtons in the y-direction. The initial state is set to zero.

In [ ]:
using DifferentialEquations

x0 = zeros(6)
f = [0.0, 500.0]
T_stop = 60.0
odefun(x, _, _) = linked_masses_ode(x, f, params)
prob = ODEProblem(odefun, x0, (0.0, T_stop))
sol = solve(prob, Tsit5())

To visualize the results, we can use the `payload_position` function. This utility function calculates the position of the payload. Let us calculate and plot the trajectory of the system:

In [ ]:
using Plots

# Vehicle position
p0_sol = hcat([u[1:2] for u in sol.u]...)
# Payload position
p_sol = hcat([payload_position(u, params) for u in sol.u]...)

# Plot in North-East frame
fig1 = plot(p0_sol[2, :], p0_sol[1, :], label="Vehicle", xlabel="East [m]", ylabel="North [m]", aspect_ratio=:equal)
plot!(fig1, p_sol[2, :], p_sol[1, :], label="Payload")
title!(fig1, "Trajectory (zero ocean current)")

As we can see, the vehicle moves east (i.e., in the y-direction). The payload is initially pointing north relative to the vehicle.

To better undersrtand the behavior of the system, we can use the `plot_linked_masses!` function. This function draws a line representing the rigid link. By adding markers to the line, we can visualize the vehicle and the payload at different times.

In [ ]:
@doc plot_linked_masses!

Let us plot the linked masses at times $t = 0, 5, 60$.

In [ ]:
plot_linked_masses!(fig1, sol(0.0), params; label="t = 0", marker=:o)
plot_linked_masses!(fig1, sol(5.0), params; label="t = 5", marker=:o)
plot_linked_masses!(fig1, sol(60.0), params; label="t = 60", marker=:o)

### Example 2

Now, let us simulate the system with a constant ocean current $V_c = [-0.2, 0]$.

In [ ]:
# Simulate
V_c = [-0.2, 0.0]
odefun_current(x, _, _) = linked_masses_ode(x, f, params, V_c)
prob_current = ODEProblem(odefun_current, x0, (0.0, T_stop))
sol_current = solve(prob_current, Tsit5())

# Vehicle position
p0_sol = hcat([u[1:2] for u in sol_current.u]...)
# Payload position
p_sol = hcat([payload_position(u, params) for u in sol_current.u]...)

# Plot in North-East frame
fig2 = plot(p0_sol[2, :], p0_sol[1, :], label="Vehicle", xlabel="East [m]", ylabel="North [m]", aspect_ratio=:equal)
plot!(fig2, p_sol[2, :], p_sol[1, :], label="Payload")
title!(fig2, "Trajectory (V_c = $(V_c))")

plot_linked_masses!(fig2, sol_current(0.0), params; label="t = 0", marker=:o)
plot_linked_masses!(fig2, sol_current(5.0), params; label="t = 5", marker=:o)
plot_linked_masses!(fig2, sol_current(60.0), params; label="t = 60", marker=:o)

Now, the ASV is drifting south and the payload is slightly offset due to the presence of currents.